In [1]:
import tensorflow as tf
import numpy as np
from TheModel import build
from sklearn.metrics import classification_report


In [2]:
## Abrir los modelos:

import os
loaded_local_models = [tf.keras.models.load_model(os.path.join(root, file)) for root, dirs, files in os.walk("./") for file in files if file.endswith('.keras')]

for i in range(len(loaded_local_models)-1):
    assert loaded_local_models[i].summary() == loaded_local_models[i+1].summary(), "Models have different architectures"

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_14 (Conv2D)          (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d_14 (MaxPoolin  (None, 13, 13, 32)       0         
 g2D)                                                            
                                                                 
 conv2d_15 (Conv2D)          (None, 11, 11, 64)        18496     
                                                                 
 max_pooling2d_15 (MaxPoolin  (None, 5, 5, 64)         0         
 g2D)                                                            
                                                                 
 flatten_7 (Flatten)         (None, 1600)              0         
                                                                 
 dense_14 (Dense)            (None, 64)               

In [3]:
train, test = tf.keras.datasets.mnist.load_data()

x_train, x_test = np.expand_dims(train[0] / 255.0, -1), np.expand_dims(test[0] / 255.0, -1)
y_train, y_test = train[1], test[1]

In [5]:
local_weights = [x.get_weights() for x in loaded_local_models]
averaged_weights = [np.median(np.array(weights), axis=0) for weights in zip(*local_weights)]

median_model = build.build_it()
median_model.set_weights(averaged_weights)

from sklearn.metrics import classification_report
# Predict the classes for the test set
y_pred = median_model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes))

median_model.save('median_model.keras')

313/313 [==============================] - 2s 5ms/step
              precision    recall  f1-score   support

           0       0.95      0.11      0.19       980
           1       0.93      0.55      0.69      1135
           2       1.00      0.01      0.02      1032
           3       0.79      0.52      0.63      1010
           4       0.20      0.98      0.34       982
           5       0.69      0.34      0.46       892
           6       1.00      0.00      0.01       958
           7       0.80      0.58      0.67      1028
           8       0.57      0.83      0.68       974
           9       0.19      0.22      0.20      1009

    accuracy                           0.42     10000
   macro avg       0.71      0.42      0.39     10000
weighted avg       0.72      0.42      0.39     10000

